# Word-Level Sentiment Analysis — Dataset Exploration

This notebook is the sentiment-analysis counterpart to the character-level exploration notebook.

We will:
- load a small labeled sentiment dataset
- inspect the raw review text and class labels
- tokenize each sentence into words
- build a word vocabulary
- encode words as integers
- pad sequences so we can batch them for PyTorch

How to read the code in this notebook:
- each code cell performs one preprocessing step
- the goal is to convert raw labeled text into tensors that a classifier can learn from
- after each step look at both the variable names and the printed outputs so the transformation stays concrete

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from nirma_university_language_models import (
    build_sentiment_dataloaders,
    build_word_vocabulary,
    encode_tokens,
    label_distribution,
    load_sentiment_examples,
    most_common_tokens,
    pad_sequence_to_length,
    tokenize_text,
)


## 1. Load the sentiment dataset

The next code cells load a small local CSV file containing review text and a sentiment label.

What students should notice:
- each example has two parts: `text` and `label`
- the labels are already categorical: `positive` or `negative`
- before building any model we should inspect a few raw examples and check class balance

In [ ]:
examples, DATA_PATH = load_sentiment_examples()

print("Dataset path:", DATA_PATH)
print("Number of examples:", len(examples))

In [ ]:
for i, example in enumerate(examples[:8], start=1):
    print(f"Example {i}: {example['label']} | {example['text']}")

print("\nLabel distribution:", label_distribution(examples))

## 2. Tokenize each sentence

Neural models do not work directly on raw strings. First we split text into tokens.

In this notebook we use a very simple tokenizer:
- lowercase the sentence
- keep alphabetic words and contractions
- return a Python list of word tokens

This is deliberately simple for teaching. The focus is on the sequence-model pipeline rather than advanced text preprocessing.

In [ ]:
sample_text = examples[0]["text"]
sample_tokens = tokenize_text(sample_text)

print("Sample text:", sample_text)
print("Tokens:", sample_tokens)

## 3. Build the word vocabulary

A vocabulary maps each token to an integer ID.

Why this matters:
- models consume numbers not words
- we reserve special tokens like `<pad>` and `<unk>`
- common words in the dataset become reusable indices across all examples

The next cells build the vocabulary and visualize the most common tokens.

In [ ]:
texts = [example["text"] for example in examples]
vocab, token_to_idx, idx_to_token = build_word_vocabulary(texts)
most_common = most_common_tokens(texts, limit=20)

print("Vocabulary size:", len(vocab))
print("First 30 vocabulary entries:", vocab[:30])
print("\nTop 20 tokens:")
print(most_common)

In [ ]:
labels = [token for token, _ in most_common]
values = [count for _, count in most_common]

plt.figure(figsize=(12, 4))
plt.bar(labels, values)
plt.title("Top 20 Most Frequent Tokens")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Encode tokens as integers

Once the vocabulary exists we can replace every token with its numeric ID.

Students should compare:
- the original sentence
- the tokenized version
- the encoded integer sequence

All three describe the same example at different stages of preprocessing.

In [ ]:
sample_ids = encode_tokens(sample_tokens, token_to_idx)

print("Sample text:", sample_text)
print("Tokens:", sample_tokens)
print("Token IDs:", sample_ids)

## 5. Where does embedding dimension come from?

For the sentiment model, the vocabulary tells us how many rows the embedding layer needs, but it does not decide the vector width.

- `vocab_size` comes from the dataset after tokenization and special tokens such as `<pad>` and `<unk>`
- `embedding_dim` is chosen by us as a hyperparameter
- each token ID is mapped to one dense vector of length `embedding_dim`
- the embedding weight matrix has shape `(vocab_size, embedding_dim)`

That means if the vocabulary has 500 tokens and we choose `embedding_dim = 32`, the model learns 500 different vectors and each vector has 32 numbers.

In [ ]:
import torch

EMBED_DIM = 32
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=EMBED_DIM)
sample_id_tensor = torch.tensor(sample_ids, dtype=torch.long)
sample_embeddings = embedding(sample_id_tensor)

print("Embedding table shape:", tuple(embedding.weight.shape))
print("Input ID shape:", tuple(sample_id_tensor.shape))
print("Output embedding shape:", tuple(sample_embeddings.shape))
print("First token embedding vector:", sample_embeddings[0])

## 6. Pad sequences for batching

Different reviews have different lengths, but a batch tensor needs a consistent shape.

The next code cell pads or truncates one sequence to a fixed maximum length.

Key ideas:
- `max_len` sets the number of time steps the model will see
- shorter sentences are padded with `<pad>` tokens
- longer sentences are truncated to keep the tensor shape uniform

In [ ]:
MAX_LEN = 12
pad_idx = token_to_idx["<pad>"]
padded_ids, original_length = pad_sequence_to_length(sample_ids, max_len=MAX_LEN, pad_idx=pad_idx)

print("Original token count:", len(sample_ids))
print("Stored length:", original_length)
print("Padded IDs:", padded_ids)
print("Recovered tokens:", [idx_to_token[idx] for idx in padded_ids])

## 7. Build DataLoaders

The final preprocessing step is to create PyTorch `DataLoader` objects.

What the next code cell does:
- converts all examples into padded integer sequences
- converts sentiment labels into class IDs
- splits the data into training and validation sets
- returns batches that the classifier can consume directly

In [ ]:
train_loader, val_loader = build_sentiment_dataloaders(
    examples,
    token_to_idx,
    max_len=MAX_LEN,
    batch_size=8,
    seed=42,
)

batch_inputs, batch_lengths, batch_labels = next(iter(train_loader))
print("Input batch shape:", batch_inputs.shape)
print("Length batch shape:", batch_lengths.shape)
print("Label batch shape:", batch_labels.shape)
print("\nFirst batch lengths:", batch_lengths.tolist())
print("First batch labels:", batch_labels.tolist())